In [1]:
from datasets import load_dataset
import polars as pl


/lfs/skampere2/0/animjha/CleanSlate/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = load_dataset("unlearning-cleanslate/cleanslate_dataset", "default")["train"].to_polars()
df_olmo = load_dataset("unlearning-cleanslate/verif-olmo-3-1125-32b-w100-s10","default")["train"].to_polars()

In [3]:
IRIS_ID = "9bd7dd3ff9165c9ecce1313aed6a2915fd8f43ab"
NEVER_GONNA_ID = "b9bb555ab4913dfeaafb409555e8f543f5180fc6"

ACTIVE_ID = IRIS_ID
reference_text = df_olmo.filter(pl.col("song_id") == ACTIVE_ID)["reference_target"].item()


In [ ]:
import requests
from tqdm.auto import tqdm

API_URL = "https://api.infini-gram-mini.io/"
INDEX = "v1_dolma-v1_7"

def igm_query(query: str, index: str = INDEX, query_type: str = "count") -> dict:
    r = requests.post(
        API_URL.rstrip("/"),
        json={"index": index, "query_type": query_type, "query": query},
        timeout=30,
    )
    r.raise_for_status()
    return r.json()

# quick sanity check
igm_query("the quick brown fox")

In [5]:
words = reference_text.split()
MAX_NGRAM = None  # set to e.g. 15 to cap the ngram length
ngrams = [
    {"ngram": " ".join(words[i:j]), "word_start": i, "word_end": j, "n_words": j - i}
    for i in range(len(words))
    for j in range(i + 2, (min(i + MAX_NGRAM, len(words)) if MAX_NGRAM else len(words)) + 1)
]
print(f"{len(ngrams)} substrings of >=2 words (from {len(words)} words)")

29890 substrings of >=2 words (from 245 words)


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def lookup(item: dict) -> dict:
    try:
        res = igm_query(item["ngram"])
        count, error = res.get("count"), None
    except Exception as e:
        count, error = None, str(e)
        print(count, error)
    return {**item, "count": count, "error": error}

with ThreadPoolExecutor(max_workers=16) as ex:
    results = list(tqdm(ex.map(lookup, ngrams), total=len(ngrams)))

res_df = pl.DataFrame(results)
res_df.sort(["n_words", "count"], descending=[True, True])

  0%|          | 2/29890 [00:00<1:18:23,  6.35it/s]

None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Fo

  0%|          | 17/29890 [00:00<16:30, 30.16it/s] 

None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/


  0%|          | 25/29890 [00:01<19:29, 25.54it/s]

...

None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/


  2%|▏         | 467/29890 [00:19<20:36, 23.80it/s]

None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/


None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/
None 403 Client Error: Forbidden for url: https://api.infini-gram-mini.io/


KeyboardInterrupt: 